# Exercise 11: foundation model embeddings for real promoters

Use the **same 251-base human non-TATA promoters and split as Exercise 9**. Label 1 means promoter; label 0 means non-promoter. Replace the learned CNN representation with embeddings from a pretrained DNA foundation model, then train logistic regression to predict the label.

Adapted from Andrea Gariboldi's [bioFoundationModels notebook](https://github.com/andrea-gariboldi/bioFoundationModels/blob/main/bioFoundation_models.ipynb), especially its tokenization and hidden-state examples. We use the same [50M Nucleotide Transformer v2](https://huggingface.co/InstaDeepAI/nucleotide-transformer-v2-50m-multi-species).

**DNA → pretrained transformer (frozen) → mean token embedding → logistic regression (trained).** Unlike Exercise 9, the feature extractor has already learned from DNA in 850 species. No earlier notebook or CSV upload is needed. **Run all includes official test evaluation.**


## Setup

In Colab, select **Runtime → Change runtime type → T4 GPU** before running. CPU execution works but extracting embeddings for all 36,131 sequences is much slower. The first run downloads the model weights and data. Lower `batch_size` if GPU memory is limited.

Install the same Transformers version as the source notebook. For local use, install the repository's `requirements.txt`. The model needs `trust_remote_code=True` to load its custom architecture; the revision below fixes both weights and model code.


In [ ]:
%pip install -q pyarrow transformers==4.57.1


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
batch_size = 32
threshold = 0.5
model_name = 'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species'
model_revision = '81b29e5786726d891dbf929404ef20adca5b36f1'

torch.manual_seed(seed)
torch.set_num_threads(2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Reuse Exercise 9's data and validation split

Download the same fixed dataset revision. Reserve 20% per label from the official training set with seed 42, giving **21,678 training / 5,419 validation / 9,034 official test sequences**. Keep the official test set unchanged and download it only in the final evaluation cell.


In [ ]:
data_url = (
    'https://huggingface.co/datasets/katarinagresova/'
    'Genomic_Benchmarks_human_nontata_promoters/resolve/'
    'e0003669df0a180c8570ebe091ed91f20fa080ec/data/'
)
benchmark_train = pd.read_parquet(data_url + 'train-00000-of-00001-aeb55728756a44ae.parquet')
benchmark_train = benchmark_train.rename(columns={'seq': 'sequence'})
assert benchmark_train['sequence'].str.fullmatch('[ACGTN]{251}').all()
assert set(benchmark_train['label']) == {0, 1}

validation_data = benchmark_train.groupby('label').sample(frac=0.20, random_state=seed)
train_data = benchmark_train.drop(validation_data.index)
y_train = train_data['label'].to_numpy()
y_validation = validation_data['label'].to_numpy()
print('Training:', len(train_data))
print('Validation:', len(validation_data))
print('Training class counts:')
print(train_data['label'].value_counts())


## 2. Load and freeze the foundation model

Pretraining asked the transformer to predict masked DNA tokens. Here we use its final hidden states as features, rather than its masked-token predictions. Each token receives a **512-dimensional contextual embedding**: its vector depends on surrounding tokens.

`requires_grad_(False)` freezes every parameter. `eval()` selects inference behavior; `torch.inference_mode()` below avoids gradient tracking. None of the promoter labels update this transformer. This is frozen feature extraction, not fine-tuning.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_name, revision=model_revision, trust_remote_code=True,
)
foundation_model = AutoModelForMaskedLM.from_pretrained(
    model_name, revision=model_revision, trust_remote_code=True,
).to(device)
foundation_model.requires_grad_(False)
foundation_model.eval()
print('Embedding dimensions:', foundation_model.config.hidden_size)
print('Trainable transformer parameters:',
      sum(p.numel() for p in foundation_model.parameters() if p.requires_grad))


## 3. Inspect DNA tokens

The tokenizer uses non-overlapping 6-mers where possible, with single bases for remaining positions and `N`. This differs from both one-hot encoding and overlapping k-mer counts. It also adds a `<cls>` token before the DNA. Inspect the actual model input below, including that special token.

Keep every base, including the official test sequence containing `N`. Use batch padding without truncation. The attention mask prevents padding from contributing to attention; a separate mask will exclude both padding and special tokens from the average.


In [ ]:
example_sequence = train_data.iloc[0]['sequence']
example_inputs = tokenizer(example_sequence, return_tensors='pt')
example_tokens = tokenizer.convert_ids_to_tokens(example_inputs['input_ids'][0].tolist())
print('DNA:', example_sequence)
print('Model tokens:', example_tokens)
print('Input shape (batch, tokens):', tuple(example_inputs['input_ids'].shape))


## 4. Extract one embedding per sequence

For each mini-batch, get final-layer token embeddings and average over DNA tokens only. This produces **one row per sequence and 512 columns**. Each DNA token has equal weight, even when it represents a single base rather than a 6-mer. Pooling compresses positional information, although each token's vector already includes context.

Move the resulting vectors to CPU memory batch by batch. Extract training and validation embeddings separately and preserve row order. No labels enter this function, and no parameters are fitted here. The test embeddings wait until final evaluation.


In [ ]:
def embed_sequences(sequences):
    sequences = list(sequences)
    batches = []
    with torch.inference_mode():
        for start in range(0, len(sequences), batch_size):
            inputs = tokenizer(
                sequences[start:start + batch_size], padding=True, truncation=False,
                return_tensors='pt',
            )
            inputs = inputs.to(device)
            outputs = foundation_model(**inputs, output_hidden_states=True)
            token_embeddings = outputs.hidden_states[-1]
            special_ids = torch.tensor(tokenizer.all_special_ids, device=device)
            special_tokens = torch.isin(inputs['input_ids'], special_ids)
            dna_mask = inputs['attention_mask'].bool() & ~special_tokens
            weights = dna_mask.unsqueeze(-1).to(token_embeddings.dtype)
            pooled = (token_embeddings * weights).sum(dim=1) / weights.sum(dim=1)
            batches.append(pooled.cpu().numpy())
            print(f'Embedded {min(start + batch_size, len(sequences))}/{len(sequences)}',
                  end='\r', flush=True)
    print()
    return np.concatenate(batches).astype(np.float32)

x_train = embed_sequences(train_data['sequence'])
x_validation = embed_sequences(validation_data['sequence'])
print('Training embeddings:', x_train.shape)
print('Validation embeddings:', x_validation.shape)


## 5. Train the promoter classifier

Standardize each embedding dimension using training-set means and standard deviations. Fit logistic regression on those scaled training embeddings. The pipeline applies the same transformation to validation and test data without fitting again. Only this scaler and classifier learn from our training set.

Use fixed regularization `C=1.0` and probability threshold 0.5. Compare validation metrics with Exercise 9 before deciding whether to change any settings. There are no transformer training epochs in this exercise.


In [ ]:
classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=seed),
)
classifier.fit(x_train, y_train)
print('Classifier input dimensions:', classifier.n_features_in_)


## 6. Evaluate validation performance

Report the same eight metrics as Exercise 9. The split is identical, so compare with its CNN results. These models use different prior training and fixed settings; neither is guaranteed to win.

Discuss: what is learned during pretraining, and what is learned here? What information might mean pooling lose? This benchmark is not chromosome-held-out, and pretraining may have included related genomic sequences. Results do not establish generalization to unseen genomes or a biological promoter mechanism.


In [ ]:
def calculate_metrics(labels, probabilities):
    predictions = (probabilities >= threshold).astype(int)
    return pd.Series({
        'accuracy': accuracy_score(labels, predictions),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'roc_auc': roc_auc_score(labels, probabilities),
        'average_precision': average_precision_score(labels, probabilities),
        'log_loss': log_loss(labels, probabilities),
    })

def plot_evaluation(labels, probabilities, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    RocCurveDisplay.from_predictions(labels, probabilities, ax=axes[0])
    axes[0].plot([0, 1], [0, 1], 'k--')
    PrecisionRecallDisplay.from_predictions(labels, probabilities, ax=axes[1])
    axes[1].axhline(np.mean(labels), color='black', linestyle='--')
    ConfusionMatrixDisplay.from_predictions(
        labels, probabilities >= threshold, labels=[0, 1],
        display_labels=['Non-promoter', 'Promoter'],
        ax=axes[2], cmap='Blues', colorbar=False,
    )
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

validation_probabilities = classifier.predict_proba(x_validation)[:, 1]
validation_results = calculate_metrics(y_validation, validation_probabilities)
print('Validation metrics:')
print(validation_results.round(4).to_string())
plot_evaluation(y_validation, validation_probabilities, 'Validation: frozen DNA embeddings')


## 7. STOP before official test evaluation

Finish all choices using validation data first. Then download the unchanged official test set, extract its embeddings with the frozen model, and evaluate the already-fitted classifier once. Do not refit the scaler or classifier, remove the sequence containing `N`, or choose settings from test results. **Run all includes this cell.**


In [ ]:
test_data = pd.read_parquet(data_url + 'test-00000-of-00001-d7614d9116c62f44.parquet')
test_data = test_data.rename(columns={'seq': 'sequence'})
assert test_data['sequence'].str.fullmatch('[ACGTN]{251}').all()
assert set(test_data['label']) == {0, 1}
x_test = embed_sequences(test_data['sequence'])
y_test = test_data['label'].to_numpy()
test_probabilities = classifier.predict_proba(x_test)[:, 1]
test_results = calculate_metrics(y_test, test_probabilities)
print('Official test sequences:', len(test_data))
print(test_results.round(4).to_string())
plot_evaluation(y_test, test_probabilities, 'Official test: frozen DNA embeddings')
